> **Legacy exploratory notebook (June 2025, first phase of the project).** Hyper-parameters were chosen on the test files and windows were split at random, so the scores it printed are not trustworthy estimates. It is kept for the record and is audited in `scripts/audit/` and `docs/legacy_audit.html`. Outputs were cleared (they contained participant file names); set `EEGBCI_LEGACY_DATA_DIR` for the helper `utils.py`.

## Import

In [ ]:
!pip install optuna

## SDXL

In [ ]:
import numpy as np
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix

from sklearn.decomposition import PCA, FastICA
from datetime import datetime
import joblib
import sys, os
import sys
import os
from pathlib import Path
# Get the current working directory
notebook_path = Path().absolute()
# Go up one directory level to reach myproject
project_root = notebook_path.parent
# Add to Python path
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
# Now you can import
from utils import read_eeg, balance_labels, make_features

# === PRESET CONFIGS ===
n_jobs = -1
cv_folds = 5

sfreq = 500
bands = {
    'alpha': (8, 12), 'beta': (12, 30)
}

best_model = None
best_acc = 0
best_conf = None

# === OBJECTIVE FUNCTION FOR OPTUNA ===
def objective(trial):
    global best_model, best_acc, best_conf  
    # print(1)
    # -------- OPTUNA PARAMETERS ----------
    win = trial.suggest_float("win_size", 0.1, 3, step=0.1)
    # print(2)
    # RANDOM FOREST HYPERPARAMETERS
    rf_params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500, step=50),
        'max_depth': trial.suggest_int('max_depth', 5, 50, step=5),
        'min_samples_split': trial.suggest_int('min_samples_split', 20, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 10, 10),
        'criterion': trial.suggest_categorical('criterion', ['gini', 'entropy']),
    }
    # print(3)

    # -------- LOAD DATA ----------
    try:
        
        X_train, y_train = read_eeg(win_size=win, why="train", verbose=False)


        X_train = make_features(X_train, verbose=False)

    except Exception as e:
        print(f"Error loading data: {e}")
        raise optuna.exceptions.TrialPruned()

    # -------- FEATURE REDUCTION ----------
    try:
        pca = PCA(n_components=0.95, svd_solver='full', random_state=1)
        X_train_pca = pca.fit_transform(X_train)
        
        ica = FastICA(n_components=min(X_train_pca.shape[1], 20), random_state=1)
        X_train = ica.fit_transform(X_train_pca)
    except:
        raise optuna.exceptions.TrialPruned()

    X_test_t = []
    y_test_t = []
    for i in range(26):
        X_test, y_test = read_eeg(win_size=win, why="test", verbose=False, who=i)
        X_test = make_features(X_test, verbose=False)
        X_test_pca = pca.transform(X_test)
        X_test = ica.transform(X_test_pca)
        X_test_t.append(X_test)
        y_test_t.append(y_test)
    
    y_test = np.concatenate(y_test_t)
    X_test = np.concatenate(X_test_t)

    print(np.unique(y_test, return_counts=True))

    # -------- CV EVALUATION ON VALIDATION SET ----------
    
    clf = RandomForestClassifier(**rf_params, random_state=1, n_jobs=n_jobs)
    clf.fit(X_train, y_train)
    y_test_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_test_pred)
    if acc > best_acc:
        best_acc = acc
        best_model = clf
    best_conf = confusion_matrix(y_test, y_test_pred)
    return acc

# === RUN OPTIMIZATION ===
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
study_name = f"rf_optuna_{timestamp}"
study = optuna.create_study(direction="maximize", study_name=study_name)
study.optimize(objective, n_trials=15)

# === RESULTS ===
print("\n=============================")
print("📊 Best Trial:")
print(f"  🏆 Accuracy: {study.best_value*100:.2f}%")
print("  🧠 Parameters:")
for k, v in study.best_params.items():
    print(f"    {k}: {v}")
print("=============================\n")


In [ ]:
!pip install plotly

In [ ]:
# ========== VISUALISATION ==========
import optuna.visualization as vis

print("📊 Génération des graphiques Optuna...")

# 1. Optimization History
vis.plot_optimization_history(study).write_html("plot_optimization_history.html", include_plotlyjs="cdn")

# 2. Parameter Importance
vis.plot_param_importances(study).write_html("plot_param_importance.html", include_plotlyjs="cdn")

# 3. Parallel Coordinates
vis.plot_parallel_coordinate(study).write_html("plot_parallel_coordinates.html", include_plotlyjs="cdn")

# 4. Slice Plot
vis.plot_slice(study).write_html("plot_slice.html", include_plotlyjs="cdn")

# 5. Contour plot for selected most relevant hyperparameters
# (You can adjust the params list depending on what was explored in the study)
contour_params = ["win_size", "n_estimators", "max_depth"]
vis.plot_contour(study, params=contour_params).write_html("plot_contour.html", include_plotlyjs="cdn")

print("✅ Plots sauvegardés en fichiers HTML.")


In [ ]:
command = [
     
        "accelerate", "launch", "train_text_to_image_lora_sdxl_train.py",
        "--pretrained_model_name_or_path=stabilityai/stable-diffusion-xl-base-1.0",
        "--pretrained_vae_model_name_or_path=madebyollin/sdxl-vae-fp16-fix",
        "--validation_data_dir=/workspace/PFE/VITON-HD-512/k_validations/",
        "--train_data_dir=/workspace/PFE/VITON-HD-512/train_images",
        "--early_stopping_patience=15",
        "--early_stopping_metric=FID",
        "--early_stopping_mode=min",
        "--caption_column=text",
        "--dataloader_num_workers=8",
        "--train_batch_size=16",
        #"--resolution=512",
        "--max_grad_norm=1",
        "--gradient_accumulation_steps=8",
        "--num_train_epochs=100",
        "--checkpointing_steps=85",
        "--lr_warmup_steps=0",
        "--mixed_precision=fp16",
        "--seed=1337",
        "--allow_tf32",
        "--use_8bit_adam",
        "--snr_gamma=5.0",
        "--enable_xformers_memory_efficient_attention",
        "--rank=8",
        "--output_dir=final_train",
        "--lr_scheduler=cosine",
        "--learning_rate=1e-5",
        "--validation_data_output=final_train/validation_images",
        "--resume_from_checkpoint=latest",
 ]
shell_command = " ".join(command)
print("lancement de l'entrainement final")
!{shell_command}

In [ ]:
!pip install bitsandbytes

In [ ]:
%ls /workspace/PFE/VITON-HD-512/k_validations/


In [ ]:
from sklearn.decomposition import PCA, FastICA
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import matplotlib.pyplot as plt
import numpy as np

# Get best parameters
best_params = study.best_params

# Load data with best window size
win_size = best_params['win_size']
X_test, y_test = read_eeg(win_size=win_size, why="test", verbose=False, who=10)  # Using test set 10 which gave best accuracy
X_test = make_features(X_test, verbose=False)

# Load training data for fitting PCA/ICA
X_train, y_train = read_eeg(win_size=win_size, why="train", verbose=False)
X_train = make_features(X_train, verbose=False)

# Apply PCA
pca = PCA(n_components=0.95, svd_solver='full', random_state=1)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

# Apply ICA
n_components = min(X_train_pca.shape[1], 20)  # Same as in training
ica = FastICA(n_components=n_components, random_state=1)
X_train_reduced = ica.fit_transform(X_train_pca)
X_test_reduced = ica.transform(X_test_pca)

# Train final model with best parameters
best_model = RandomForestClassifier(
    n_estimators=best_params['n_estimators'],
    max_depth=best_params['max_depth'],
    min_samples_split=best_params['min_samples_split'],
    min_samples_leaf=best_params['min_samples_leaf'],
    criterion=best_params['criterion'],
    random_state=1,
    n_jobs=-1
)

# Fit on all training data
best_model.fit(X_train_reduced, y_train)

# Make predictions
y_pred = best_model.predict(X_test_reduced)

# Generate confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                            display_labels=np.unique(np.concatenate([y_test, y_pred])))
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix for Best Model')
# plt.show()

# Print classification report
# print("Classification Report:")
# print(classification_report(y_test, y_pred))

# Print accuracy
accuracy = np.mean(y_test == y_pred)
print(f"Best Accuracy: {accuracy*100:.2f}%")
